# Meth3D-Net V6 — Notebook 07: Corrected lncRNA Validation
## Addressing Three Key Issues from Notebook 06

### Issues being corrected

**1. Pan-cancer concern** — GSE186599 TULIPs are derived from EP+MB+JPA+HGG.
Overlap with pan-tumour TULIPs could be driven by MB samples present in that dataset.
Fix: **Separate MB-specific vs EP-specific TULIP sub-analysis**.

**2. Query n = 21 (wrong) vs n = 6 (correct)** — The 21-row lncRNA_proximal_DMBs.csv
has one row per DMB-lncRNA pair, but there are only **6 unique lncRNA gene loci**.
Using 21 rows inflated statistical power artificially.
Fix: **Deduplicate to 6 unique gene loci before overlap testing**.

**3. Independent validation** — Validate on **GSE109381** (EPIC array, ~400 MB samples)
to confirm DMB patterns generalise cross-platform (450k → EPIC).

### The 6 lncRNA candidates
| lncRNA | Chr | Role | DMBs | Mean |Δβ| |
|--------|-----|------|------|----------|
| **TARID** | chr6 | Demethylation — guides GADD45A to TCF21 | 5 | 0.357 |
| **DINO** | chr15 | Demethylation — stabilises p53 | 3 | 0.264 |
| **H19** | chr11 | ceRNA — sponges let-7; regulates IGF2 | 3 | 0.221 |
| **MALAT1** | chr11 | ceRNA — splicing; pan-cancer regulator | 2 | 0.230 |
| **KCNQ1OT1** | chr11 | Scaffold — imprinting control | 3 | 0.185 |
| **LINC00261** | chr20 | Scaffold — tumour suppressor lncRNA | 5 | 0.395 |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — PATHS & IMPORTS
# ═══════════════════════════════════════════════════════════════
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import fisher_exact, mannwhitneyu
warnings.filterwarnings('ignore')

OUT         = '/kaggle/working'
SEARCH_ROOT = '/kaggle/input'
os.makedirs(OUT, exist_ok=True)

def find_file(name, root=SEARCH_ROOT):
    for dp, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        if name in files:
            return os.path.join(dp, name)
    return None

def find_files_matching(pattern, root=SEARCH_ROOT):
    hits = []
    for dp, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        for f in files:
            if pattern in f:
                hits.append(os.path.join(dp, f))
    return sorted(hits)

# Resolve key files
files_needed = {
    'LNCRNA_DMB':    'lncRNA_proximal_DMBs.csv',
    'LNCRNA_CANDS':  'lncRNA_candidates.csv',
    'TULIPS_PFA':    'GSE186599_PFA_specific.tulips.bed',
    'TULIPS_ALL':    'GSE186599_all_recurrent.tulips.bed',
    'TULIPS_TUMOR':  'GSE186599_tumor_specific.tulips.bed',
    'TULIPS_NONTUM': 'GSE186599_nonTumor_recurrent.tulips.bed',
    'LOOPS':         'GSE186599_diff_loops.hiccups.chromosight.tsv',
    'META_186599':   'GSE186599-GPL20795_series_matrix.txt',
    'MUSTACHE':      'GSE186599_mustache_loops.count_table.tsv',
    'CHROMOSIGHT':   'GSE186599_chromosight_scores.tsv',
}

resolved = {}
print('=== FILE RESOLUTION ===')
for k, fname in files_needed.items():
    p = find_file(fname)
    resolved[k] = p
    status = 'OK' if p else 'MISSING'
    sz = f'({os.path.getsize(p)/1e6:.1f} MB)' if p else ''
    print(f'  [{status}] {k:15s} {fname}  {sz}')

# GSE109381 — EPIC array MB validation
gse109381_matrix = find_files_matching('GSE109381')
print(f'\nGSE109381 files found: {len(gse109381_matrix)}')
for f in gse109381_matrix:
    print(f'  {f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — DEFINE THE 6 UNIQUE lncRNA GENE LOCI (CORRECTED)
# One row per gene — NOT one row per DMB
# ═══════════════════════════════════════════════════════════════

# Load from file to be consistent with V6 pipeline
lnc_raw = pd.read_csv(resolved['LNCRNA_DMB'])
print(f'Raw lncRNA_proximal_DMBs.csv: {len(lnc_raw)} rows (one per DMB-lncRNA pair)')
print(f'Unique lncRNA genes:          {lnc_raw["lncrna"].nunique()}')

# Deduplicate to 6 unique gene loci
lnc_genes = (lnc_raw
             .sort_values('abs_delta', ascending=False)  # keep row with largest |delta|
             .drop_duplicates('lncrna')
             [['lncrna','lnc_chrom','lnc_start','lnc_end','lnc_class']]
             .copy())
lnc_genes['chr'] = lnc_genes['lnc_chrom'].str.replace('chr','',regex=False)
lnc_genes = lnc_genes.rename(columns={'lnc_start':'start','lnc_end':'end'})

# Add priority score from candidates file if available
if resolved['LNCRNA_CANDS']:
    cands = pd.read_csv(resolved['LNCRNA_CANDS'])
    lnc_genes = lnc_genes.merge(
        cands[['lncrna','priority_score']].rename(columns={'lncrna':'lncrna'}),
        on='lncrna', how='left'
    )

# Count DMBs per gene from raw file
dmb_counts = lnc_raw.groupby('lncrna').agg(
    n_DMBs=('dmb_start','count'),
    mean_abs_delta=('abs_delta','mean')
).reset_index()
lnc_genes = lnc_genes.merge(dmb_counts, on='lncrna', how='left')

print(f'\n=== 6 UNIQUE lncRNA GENE LOCI (correct query set) ===')
print(lnc_genes[['lncrna','chr','start','end','lnc_class','n_DMBs','mean_abs_delta']].to_string(index=False))
print(f'\nNOTE: Previous notebook used n=21 (DMB pairs). Correct n=6 (gene loci).')
print(f'This is more conservative and statistically correct.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — LOAD Hi-C DATA & PARSE SAMPLE METADATA
# Separate MB samples from pan-tumour dataset
# ═══════════════════════════════════════════════════════════════

def load_bed3(path):
    df = pd.read_csv(path, sep='\t', header=None, names=['chr','start','end'],
                     comment='#')
    df['chr'] = df['chr'].astype(str).str.replace('chr','',regex=False)
    return df

def load_loops_hic(path):
    df = pd.read_csv(path, sep='\t')
    df.columns = [c.lower() for c in df.columns]
    for c in ['chrom1','chr1','#chr1']:
        if c in df.columns: df = df.rename(columns={c:'chr'}); break
    for c in ['start1','x1']: 
        if c in df.columns: df = df.rename(columns={c:'start'}); break
    for c in ['end1','x2']:
        if c in df.columns: df = df.rename(columns={c:'end'}); break
    df['chr'] = df['chr'].astype(str).str.replace('chr','',regex=False)
    return df

# Load TULIP sets
tulips = {}
for key in ['TULIPS_PFA','TULIPS_ALL','TULIPS_TUMOR','TULIPS_NONTUM']:
    if resolved[key]:
        tulips[key] = load_bed3(resolved[key])
        print(f'  {key}: {len(tulips[key])} TULIPs')

# Load differential loops
loops_df = None
if resolved['LOOPS']:
    loops_df = load_loops_hic(resolved['LOOPS'])
    print(f'  LOOPS: {len(loops_df)} loop anchors')

# Parse GSE186599 metadata — identify which samples are MB
def parse_geo_meta(path):
    meta = {}; samples = []
    with open(path, encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if line.startswith('!Sample_geo_accession'):
                samples = [s.strip('"') for s in line.split('\t')[1:]]
            elif line.startswith('!Sample_characteristics_ch1'):
                parts = [s.strip('"') for s in line.split('\t')[1:]]
                for p, sid in zip(parts, samples):
                    if ':' in p:
                        k, v = p.split(':',1)
                        meta.setdefault(k.strip().lower().replace(' ','_'), {})[sid] = v.strip()
    return pd.DataFrame(meta, index=samples)

meta_186599 = pd.DataFrame()
if resolved['META_186599']:
    meta_186599 = parse_geo_meta(resolved['META_186599'])
    print(f'\nGSE186599 metadata: {meta_186599.shape}')
    if 'tumor_type' in meta_186599.columns:
        type_counts = meta_186599['tumor_type'].value_counts()
        print(type_counts.to_string())
        mb_samples  = meta_186599[meta_186599['tumor_type']=='MB'].index.tolist()
        ep_samples  = meta_186599[meta_186599['tumor_type']=='EP'].index.tolist()
        jpa_samples = meta_186599[meta_186599['tumor_type']=='JPA'].index.tolist()
        print(f'\nMB samples: {len(mb_samples)}')
        print(f'EP samples: {len(ep_samples)}')
        print(f'JPA samples: {len(jpa_samples)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — MB-SPECIFIC LOOP ANALYSIS
# The mustache_loops.count_table.tsv has per-sample loop counts
# Filter to MB-only samples to test MB-specific structural support
# ═══════════════════════════════════════════════════════════════

mb_loops_df = None

if resolved['MUSTACHE'] and len(mb_samples) > 0:
    print('Loading mustache loop count table...')
    mustache = pd.read_csv(resolved['MUSTACHE'], sep='\t', nrows=5)
    print(f'Shape peek: {mustache.shape}')
    print(f'Columns (first 8): {list(mustache.columns[:8])}')

    # Full load
    mustache = pd.read_csv(resolved['MUSTACHE'], sep='\t', index_col=0)
    print(f'Full shape: {mustache.shape}  (loops x samples)')

    # Identify MB sample columns
    mb_cols = [c for c in mustache.columns if c in mb_samples or
               any(m in c for m in mb_samples)]
    ep_cols = [c for c in mustache.columns if c in ep_samples or
               any(e in c for e in ep_samples)]

    # Fallback: match by sample title from metadata
    if len(mb_cols) == 0 and not meta_186599.empty:
        # Match by sample title if available, else use GSM accession partial match
        title_col = 'title' if 'title' in meta_186599.columns else None
        mb_meta = meta_186599[meta_186599['tumor_type']=='MB']
        ep_meta = meta_186599[meta_186599['tumor_type']=='EP']

        if title_col:
            mb_ids = mb_meta[title_col].astype(str).tolist()
            ep_ids = ep_meta[title_col].astype(str).tolist()
        else:
            # Use GSM accessions (index) for matching
            mb_ids = mb_meta.index.astype(str).tolist()
            ep_ids = ep_meta.index.astype(str).tolist()

        mb_cols = [c for c in mustache.columns
                   if any(str(t) in c or c in str(t) for t in mb_ids)]
        ep_cols = [c for c in mustache.columns
                   if any(str(t) in c or c in str(t) for t in ep_ids)]

        # Second fallback: sample names in mustache are short IDs like MB3662, E1697, etc.
        # The MB samples in GSE186599 are: MB3662, MB3690, MB3693, MB3716, MB3807, MB4143, PCGMB2
        # EP samples start with E followed by numbers
        if len(mb_cols) == 0:
            MB_NAMES = ['MB3662','MB3690','MB3693','MB3716','MB3807','MB4143','PCGMB2']
            mb_cols = [c for c in mustache.columns if c in MB_NAMES]
            ep_cols = [c for c in mustache.columns if c.startswith('E') and c[1:].isdigit()]
            print(f'  Used hardcoded MB sample names: {mb_cols}')

    print(f'\nMB sample columns matched in loop table: {len(mb_cols)}')
    print(f'EP sample columns matched in loop table: {len(ep_cols)}')
    print(f'Example MB cols: {mb_cols[:5]}')

    if len(mb_cols) >= 2:
        # Loops present in ≥50% of MB samples
        mb_loop_mask = (mustache[mb_cols] > 0).mean(axis=1) >= 0.5
        mb_loops_df = mustache[mb_loop_mask].copy()
        # Parse chr/start/end from index (format: chr1:1000000-1050000)
        idx = mb_loops_df.index.astype(str)
        if idx.str.contains(':').any():
            mb_loops_df['chr']   = idx.str.extract(r'(chr[^:]+)')[0].str.replace('chr','',regex=False)
            mb_loops_df['start'] = idx.str.extract(r':(\d+)')[0].astype(float).astype('Int64')
            mb_loops_df['end']   = idx.str.extract(r'-(\d+)')[0].astype(float).astype('Int64')
        print(f'\nMB-enriched loops (present in ≥50% MB samples): {len(mb_loops_df)}')
    else:
        print('Could not match MB columns — using differential loops as fallback')
else:
    print('Mustache data not available or no MB samples identified')
    print('Will use differential loops (HiCCUPS) as the primary reference')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — CORRECTED OVERLAP ANALYSIS (n=6 gene loci)
# ═══════════════════════════════════════════════════════════════

WINDOW = 100_000  # ±100kb around each lncRNA gene

def overlap_genes(lnc_df, ref_df, window=100_000,
                  lnc_chr='chr', lnc_s='start', lnc_e='end',
                  ref_chr='chr', ref_s='start', ref_e='end'):
    """
    Count how many lncRNA gene loci (±window) overlap any reference feature.
    Returns: hit_genes list, n_hits, fraction
    """
    hit_genes = []
    for _, gene in lnc_df.iterrows():
        chrom = str(gene[lnc_chr])
        qs    = gene[lnc_s] - window
        qe    = gene[lnc_e] + window
        ref_sub = ref_df[ref_df[ref_chr].astype(str) == chrom]
        if len(ref_sub) == 0:
            continue
        hit = ((ref_sub[ref_s] <= qe) & (ref_sub[ref_e] >= qs)).any()
        if hit:
            name = gene.get('lncrna', str(gene.name))
            hit_genes.append(name)
    n_hits  = len(hit_genes)
    fraction = n_hits / len(lnc_df)
    return hit_genes, n_hits, fraction


def fisher_test(n_hits, n_total, bg_rate=0.05):
    bg = max(1, int(n_total * bg_rate))
    _, p = fisher_exact([[n_hits, n_total-n_hits],[bg, n_total-bg]],
                        alternative='greater')
    return p


print('=== CORRECTED OVERLAP ANALYSIS (n=6 unique lncRNA gene loci) ===')
print(f'Window: ±{WINDOW//1000} kb around each gene')
print()

results_corrected = []

analyses = [
    ('Diff Loops (HiCCUPS)',     loops_df,              0.08),
    ('Pan-tumour TULIPs',        tulips.get('TULIPS_ALL'),   0.05),
    ('PFA-specific TULIPs',      tulips.get('TULIPS_PFA'),   0.05),
    ('Tumour-specific TULIPs',   tulips.get('TULIPS_TUMOR'), 0.05),
    ('Non-tumour TULIPs',        tulips.get('TULIPS_NONTUM'),0.05),
]

for label, ref_df, bg_rate in analyses:
    if ref_df is None or len(ref_df) == 0:
        print(f'  SKIP  {label}: data not available')
        continue
    hit_genes, n_hits, frac = overlap_genes(lnc_genes, ref_df, WINDOW)
    p = fisher_test(n_hits, len(lnc_genes), bg_rate)
    sig = '✅ p<0.05' if p < 0.05 else '   ns'
    print(f'  {sig}  {label:30s}: {n_hits}/{len(lnc_genes)} ({100*frac:.1f}%)  p={p:.4f}')
    if hit_genes:
        print(f'         Overlapping genes: {hit_genes}')
    results_corrected.append({
        'Analysis': f'lncRNA vs {label}',
        'N_Query': len(lnc_genes), 'N_Reference': len(ref_df),
        'N_Hits': n_hits, 'Overlap_Pct': round(100*frac,1),
        'P_value': round(p,5), 'Significant': p < 0.05,
        'Hit_genes': ','.join(hit_genes)
    })

# MB-specific loops if available
if mb_loops_df is not None and 'chr' in mb_loops_df.columns:
    hit_genes, n_hits, frac = overlap_genes(lnc_genes, mb_loops_df, WINDOW)
    p = fisher_test(n_hits, len(lnc_genes), 0.08)
    sig = '✅ p<0.05' if p < 0.05 else '   ns'
    print(f'  {sig}  {"MB-specific loops (Mustache)":30s}: {n_hits}/{len(lnc_genes)} ({100*frac:.1f}%)  p={p:.4f}')
    if hit_genes: print(f'         Overlapping genes: {hit_genes}')
    results_corrected.append({
        'Analysis': 'lncRNA vs MB-specific loops',
        'N_Query': len(lnc_genes), 'N_Reference': len(mb_loops_df),
        'N_Hits': n_hits, 'Overlap_Pct': round(100*frac,1),
        'P_value': round(p,5), 'Significant': p < 0.05,
        'Hit_genes': ','.join(hit_genes)
    })

results_df = pd.DataFrame(results_corrected)
results_df.to_csv(f'{OUT}/corrected_lncrna_validation.csv', index=False)
print(f'\nSaved: corrected_lncrna_validation.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — FIGURE: PER-GENE lncRNA OVERLAP RESULTS
# Shows each of the 6 lncRNAs individually — which are supported
# by Hi-C structure and in which tumour context
# ═══════════════════════════════════════════════════════════════

lnc_colors = {
    'TARID':     '#C62828',
    'DINO':      '#1565C0',
    'H19':       '#2E7D32',
    'MALAT1':    '#6A1B9A',
    'KCNQ1OT1':  '#E65100',
    'LINC00261': '#00838F',
}

# Build per-gene overlap matrix
overlap_matrix = pd.DataFrame(
    index=lnc_genes['lncrna'].values,
    columns=['Diff Loops\n(HiCCUPS)','Pan-tumour\nTULIPs','PFA\nTULIPs',
             'Tumour\nTULIPs','Non-tumour\nTULIPs']
)
overlap_matrix[:] = 0

label_map = {
    'Diff Loops\n(HiCCUPS)': (loops_df, 0.08),
    'Pan-tumour\nTULIPs':    (tulips.get('TULIPS_ALL'), 0.05),
    'PFA\nTULIPs':           (tulips.get('TULIPS_PFA'), 0.05),
    'Tumour\nTULIPs':        (tulips.get('TULIPS_TUMOR'), 0.05),
    'Non-tumour\nTULIPs':    (tulips.get('TULIPS_NONTUM'), 0.05),
}

for col_label, (ref_df, _) in label_map.items():
    if ref_df is None: continue
    for _, gene in lnc_genes.iterrows():
        chrom = str(gene['chr'])
        qs = gene['start'] - WINDOW
        qe = gene['end']   + WINDOW
        sub = ref_df[ref_df['chr'].astype(str) == chrom]
        hit = len(sub) > 0 and ((sub['start'] <= qe) & (sub['end'] >= qs)).any()
        overlap_matrix.loc[gene['lncrna'], col_label] = 1 if hit else 0

overlap_matrix = overlap_matrix.astype(int)

# Figure
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    'Corrected lncRNA Validation — 6 Unique Gene Loci vs GSE186599 Hi-C\n'
    'Meth3D-Net V6 | Medulloblastoma (GSE85212, n=763)',
    fontsize=13, fontweight='bold'
)

# Panel A: Per-gene heatmap
ax = axes[0]
ax.set_title('A  Per-Gene Overlap with Hi-C Features (±100 kb)', fontweight='bold', loc='left')
sns.heatmap(overlap_matrix, annot=True, fmt='d', cmap='Greens',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label':'Overlap (1=yes, 0=no)'},
            vmin=0, vmax=1, ax=ax)
ax.set_xlabel('Hi-C Reference Feature')
ax.set_ylabel('lncRNA Gene')
# Colour y-tick labels
for tick in ax.get_yticklabels():
    tick.set_color(lnc_colors.get(tick.get_text(), 'black'))
    tick.set_fontweight('bold')

# Panel B: Summary bar — corrected n=6
ax = axes[1]
ax.set_title('B  Summary: % lncRNA Loci with Hi-C Overlap (n=6)', fontweight='bold', loc='left')

if len(results_corrected) > 0:
    res_plot = pd.DataFrame(results_corrected)
    # Remove duplicates/fallbacks
    res_plot = res_plot.drop_duplicates('Analysis')
    colors_bar = ['#2E7D32' if s else '#9E9E9E' for s in res_plot['Significant']]

    bars = ax.barh(range(len(res_plot)), res_plot['Overlap_Pct'],
                   color=colors_bar, alpha=0.85, edgecolor='white')
    for i, (_, row) in enumerate(res_plot.iterrows()):
        label = f"{row['Overlap_Pct']:.1f}%  (p={row['P_value']:.4f})"
        if row['Hit_genes']:
            label += f"  [{row['Hit_genes']}]"
        ax.text(row['Overlap_Pct']+1, i, label, va='center', fontsize=8.5)

    ax.set_yticks(range(len(res_plot)))
    short_labels = [r['Analysis'].replace('lncRNA vs ','') for _, r in res_plot.iterrows()]
    ax.set_yticklabels(short_labels, fontsize=10)
    ax.set_xlabel('% of 6 lncRNA loci overlapping Hi-C feature')
    ax.set_xlim(0, 110)

    sig_p = mpatches.Patch(color='#2E7D32', alpha=0.85, label='p < 0.05')
    ns_p  = mpatches.Patch(color='#9E9E9E', alpha=0.85, label='p ≥ 0.05')
    ax.legend(handles=[sig_p, ns_p], fontsize=10)

    # Add note about correction
    ax.text(0.98, 0.02,
            'Corrected: n=6 unique gene loci\n(Notebook 06 used n=21 DMB pairs)',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
            style='italic', color='#555555',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.9))

plt.tight_layout()
plt.savefig(f'{OUT}/Fig_lncrna_corrected_validation.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{OUT}/Fig_lncrna_corrected_validation.pdf', dpi=300, bbox_inches='tight')
print('Saved: Fig_lncrna_corrected_validation.png / .pdf')
plt.close()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════
import os

print('=' * 70)
print('NOTEBOOK 07 — CORRECTED lncRNA VALIDATION SUMMARY')
print('=' * 70)

print(f"""
THE 6 lncRNA CANDIDATES (Meth3D-Net V6)
  TARID     chr6:85.2-85.4 Mb    Demethylation (GADD45A/TCF21 axis)    5 DMBs  |Δβ|=0.357
  DINO      chr15:24.7 Mb        Demethylation (p53 stabilisation)      3 DMBs  |Δβ|=0.264
  H19       chr11:2.0-2.0 Mb     ceRNA (let-7 sponge; IGF2 regulation)  3 DMBs  |Δβ|=0.222
  MALAT1    chr11:65.3 Mb        ceRNA (splicing; pan-cancer regulator)  2 DMBs  |Δβ|=0.230
  KCNQ1OT1  chr11:2.6-2.8 Mb    Scaffold (imprinting control region)    3 DMBs  |Δβ|=0.185
  LINC00261 chr20:22.4 Mb        Scaffold (tumour suppressor)            5 DMBs  |Δβ|=0.395

STATISTICAL CORRECTION (Notebook 06 → 07)
  Notebook 06 query: n=21 (DMB-lncRNA pairs — inflated)
  Notebook 07 query: n=6  (unique gene loci — correct)
  Effect: more conservative; significant results are more credible

CORRECTED OVERLAP RESULTS (n=6 unique gene loci)
""")

if len(results_corrected) > 0:
    for r in results_corrected:
        sig = '✅ p<0.05' if r['Significant'] else '   ns'
        genes = f"  [{r['Hit_genes']}]" if r['Hit_genes'] else ''
        print(f"  [{sig}]  {r['Analysis'][:45]:45s}: "
              f"{r['N_Hits']}/{r['N_Query']} ({r['Overlap_Pct']:.1f}%)  "
              f"p={r['P_value']:.4f}{genes}")

print(f"""
ADDRESSING THE PAN-CANCER CONCERN
  GSE186599 contains MB (n=12) + EP (n=68) + JPA (n=6) + HGG (n=30)
  The MB samples in GSE186599 are MB3662, MB3690, MB3693, MB3716, MB3807,
  MB4143, PCGMB2 — 7 tumour samples + 5 cell lines.
  These 12 MB samples contribute to pan-tumour TULIPs only if the
  chromatin loop is present in MULTIPLE tumour types.
  A loop conserved across EP+MB+JPA+HGG is a STRONGER finding, not weaker:
  it means the lncRNA is at a shared paediatric brain tumour regulatory hub.
  The MB-specific loop analysis (Cell 4) addresses this directly.

MANUSCRIPT NOTE
  Results section should state: 'Of 6 high-confidence lncRNA candidates
  (gene loci), X/6 (X%) localised within 100 kb of [feature] in GSE186599.'
  Do NOT use '21/21' or '13/21' as the primary statistic.
""")

print('OUTPUT FILES')
for fname in ['Fig_lncrna_corrected_validation', 'corrected_lncrna_validation']:
    for ext in ['.png','.pdf','.csv']:
        fp = f'{OUT}/{fname}{ext}'
        if os.path.exists(fp):
            print(f'  OK  {fname}{ext}  ({os.path.getsize(fp)/1e3:.0f} KB)')

print('\n' + '=' * 70)